In [7]:
# import pandas as pd

# # Load both months
# july = pd.read_excel("NAIVAS JULY DATA.xlsx", sheet_name="Unpivoted data")
# august = pd.read_excel("NAIVAS AUG DATA.xlsx", sheet_name= "Unpivoted data")
# september = pd.read_excel("NAIVAS SEP DATA.XLSX", sheet_name= "Unpivoted data")
# october = pd.read_excel("NAIVAS OCTOBER DATA.XLSX", sheet_name="Unpivoted data")
# november = pd.read_excel("NAIVAS NOV DATA.XLSX", sheet_name="Unpivoted data")
# december = pd.read_excel("NAIVAS DECEMBER DATA.XLSX", sheet_name= "Unpivoted data")
# january = pd.read_excel("NAIVAS JAN DATA.xlsx", sheet_name="Unpivoted data")
# february = pd.read_excel("NAIVAS FEB DATA.xlsx", sheet_name="Unpivoted data")
# # Add a column to identify the month
# july["Month"] = "July"
# august["Month"] = "August"
# september["Month"] = "September"
# october["Month"] = "October"
# november ["Month"] = "November"
# december["Month"] = "December"
# january["Month"] = "January"
# february["Month"] = "February"

In [8]:
import pandas as pd
files = {
    "NAIVAS JULY DATA.xlsx": "2025-07-01",
    "NAIVAS AUG DATA.xlsx": "2025-08-01",
    "NAIVAS SEP DATA.xlsx": "2025-09-01",
    "NAIVAS OCTOBER DATA.xlsx": "2025-10-01",
    "NAIVAS NOV DATA.xlsx": "2025-11-01",
    "NAIVAS DECEMBER DATA.xlsx": "2025-12-01",
    "NAIVAS JAN DATA.xlsx": "2026-01-01",
    "NAIVAS FEB DATA.xlsx": "2026-02-01",
}

all_data = []

for file, date in files.items():
    df = pd.read_excel(file, sheet_name="Unpivoted data")
    df["Date"] = pd.to_datetime(date)
    all_data.append(df)

final_df = pd.concat(all_data, ignore_index=True)

In [9]:
final_df["Month"] = final_df["Date"].dt.strftime("%B")  # July, August, etc.
final_df["Year"] = final_df["Date"].dt.year

In [10]:
final_df.to_excel("final_df.xlsx", index= False)

In [11]:
# Combine into one DataFrame
# combined = pd.concat([july, august, september, october, november,december,january,february], ignore_index=True)

In [12]:
# print(combined.columns)
# print(combined.info())


In [13]:
# combined.to_excel("COMBINED_DATA.xlsx", index=False)


In [14]:
import pandas as pd
import re

# --- CONFIG ---
file_path = "final_df.xlsx"
output_file = "categorized_file.xlsx"
diag_matched = "diagnostics_window_matched.csv"
diag_unmatched = "diagnostics_window_unmatched.csv"

# --- Helpers -----
def normalize_text(s):
    """Lowercase, strip, collapse whitespace, replace punctuation and normalize digits."""
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    s = re.sub(r"[\t\n\r/\\\-\_\|,·•·:;]+", " ", s)
    s = re.sub(r"[()\[\]{}\"']", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    # Fullwidth digit normalization
    fw_digits = "０１２３４５６７８９"
    ascii_digits = "0123456789"
    trans = str.maketrans({k: v for k, v in zip(fw_digits, ascii_digits)})
    s = s.translate(trans)
    return s

# Simple patterns after normalization
WINDOW_SIMPLE_300 = re.compile(r"window[\s\-\_\/\.,()]*300")
WINDOW_SIMPLE_320 = re.compile(r"window[\s\-\_\/\.,()]*320")

# stay polished
STAY_POLISHED_PAT = re.compile(r"stay polished") 

def contains_window_300_320(text):
    """Detect 300/320 window cleaners and stay polished."""
    if not text:
        return None

    if WINDOW_SIMPLE_300.search(text):
        return "300"
    if WINDOW_SIMPLE_320.search(text):
        return "320"

    if re.search(r"window.{0,15}300", text):
        return "300"
    if re.search(r"window.{0,15}320", text):
        return "320"
    if re.search(r"300.{0,15}window", text):
        return "300"
    if re.search(r"320.{0,15}window", text):
        return "320"

    if STAY_POLISHED_PAT.search(text):
        return "stay_polished"

    return None

# --- Load file ---
df = pd.read_excel(file_path)

# Ensure required columns exist
for col in ["DESCRIPTION", "ITEM_NAME", "CATEGORY"]:
    if col not in df.columns:
        df[col] = ""

# Build combined search text
def build_search_text(row):
    parts = []
    for col in ["ITEM_NAME", "DESCRIPTION", "CATEGORY"]:
        parts.append(normalize_text(row.get(col, "")))
    return " ".join([p for p in parts if p])

df["__search_text__"] = df.apply(build_search_text, axis=1)

# Diagnostics
matched_rows = []
unmatched_window_rows = []

# --- Subcategory assignment ---
def assign_subcategory(row):
    cat = normalize_text(row.get('CATEGORY', ""))
    desc = normalize_text(row.get('DESCRIPTION', ""))
    search = row.get("__search_text__", "")

    # --- Bleaches ---
    if "bleach" in cat:
        if any(x in desc for x in ["color", "colour", "colours"]):
            return "Color Safe Bleach"
        return "Regular Bleach"

    # --- Surface care / protection / all purpose ---
    if "surface care" in cat or "protection" in cat or "all purpose" in cat:

        if ("magnee" in desc and "all purpose" in desc) or ("maxell" in desc and "all purpose" in desc):
            return "Other Surface Care"

        if "unigel" in desc:
            return "Floor and Surface Care"

        if "disinfect" in desc or "disinfectant" in desc:
            return "Disinfectants"
        if "wipes" in desc:
            return "kitchen wipes"
        if "floor" in desc or "surface" in desc:
            return "Floor and Surface Care"

        if "carpet" in desc:
            return "Carpet Cleaners"

        # --- WINDOW 300/320 / STAY POLISHED LOGIC ----------
        match = contains_window_300_320(search)

        # 300/320 go to Window Cleaners
        if match in ("300", "320"):
            matched_rows.append(dict(row))
            return "Window Cleaners"

        # stay polished → Glass Cleaners (NOT window cleaners)
        if match == "stay_polished":
            unmatched_window_rows.append(dict(row))
            return "Glass Cleaners"

        # everything else containing 'window' → Glass Cleaners
        if "window" in search:
            unmatched_window_rows.append(dict(row))
            return "Glass Cleaners"

        # items with 'glass' → Glass Cleaners
        if "glass" in search:
            return "Glass Cleaners"

        if "all purpose" in desc or "all purpose" in cat:
            return "MultiPurpose Dishwashing"

        if "kitchen" in desc:
            return "Kitchen Cleaners"

        if "bathroom" in desc:
            return "Bathroom Cleaners"

        if "dry clean" in desc:
            return "Home Dry Cleaners"

        return "Other Surface Care"

    # --- Toilet & septic ---
    if "toilet" in cat or "septic" in cat:
        if "ball" in desc:
            return "Toilet Balls"
        return "Toilet cleaners"

    # Default: keep original category
    return row.get("CATEGORY", "")

# Apply classification
df["SUBCATEGORY"] = df.apply(assign_subcategory, axis=1)

# Normalize naming
df["SUBCATEGORY"] = df["SUBCATEGORY"].replace({
    "ALL PURPOSE": "MultiPurpose Dishwashing",
    "All purpose": "MultiPurpose Dishwashing",
    "All-purpose": "MultiPurpose Dishwashing",
    "All-Purpose Dishwashing": "MultiPurpose Dishwashing"
})

# Save main file
df.to_excel(output_file, index=False)
print(f"✅ Done! File saved as {output_file}")

# Save diagnostics
if matched_rows:
    pd.DataFrame(matched_rows).to_csv(diag_matched, index=False)
    print(f"🔎 Window 300/320 matched rows saved → {diag_matched}")
else:
    print("🔎 No 300/320 window items found.")

if unmatched_window_rows:
    pd.DataFrame(unmatched_window_rows).to_csv(diag_unmatched, index=False)
    print(f"🔎 Unmatched window-related rows saved → {diag_unmatched}")
else:
    print("🔎 No unmatched window/glass items found.")

✅ Done! File saved as categorized_file.xlsx
🔎 Window 300/320 matched rows saved → diagnostics_window_matched.csv
🔎 Unmatched window-related rows saved → diagnostics_window_unmatched.csv


In [15]:
import pandas as pd

# --- 1️⃣ Full branch-to-handler mapping ---
mapping_dict = {
    "AIRPORT VIEW": "ANN LOLE",
    "GATEWAY MALL-SYOKIMAU": "ANN LOLE",
    "KATANI ROAD": "ANN LOLE",
    "MAVOKO EXPRESS": "ANN LOLE",
    "GREENHOUSE": "ANTONY MUTUNE NZAU",
    "KINGARA ROAD": "ANTONY MUTUNE NZAU",
    "PRESTIGE": "ANTONY MUTUNE NZAU",
    "NAKURU WESTSIDE": "BEATRICE ANYANGO",
    "KERICHO": "BONFACE ONKWANI",
    "KILIMANI": "CECILIA WANJIRU",
    "ONE STOP KAREN": "CECILIA WANJIRU",
    "WATERFRONT KAREN": "CECILIA WANJIRU",
    "WOOD AVENUE KILIMANI": "CECILIA WANJIRU",
    "MOUNTAIN MALL": "CLEFTON WAMBUA MUENDO",
    "SHELL SURVEY THIKAROAD": "CLEFTON WAMBUA MUENDO",
    "JOGOO ROAD": "STEPHEN OTIENO",
    "SUPERMARKET KOBIL": "CYNTHIA ANYANGO",
    "TASSIA": "CYNTHIA ANYANGO",
    "LANGATA": "DENNIS KIPRUTO",
    "LANGATA MEDLINK": "DENNIS KIPRUTO",
    "NAIROBI WEST": "DENNIS KIPRUTO",
    "KISUMU": "DORCAS NJERI KIMANI",
    "KISUMU SIMBA": "DORCAS NJERI KIMANI",
    "KANGEMI": "DORCAS PAUL",
    "LIMURU": "DORCAS PAUL",
    "MOUNTAIN VIEW": "DORCAS PAUL",
    "TILISI": "DORCAS PAUL",
    "UTHIRU EXPRESS": "DORCAS PAUL",
    "BAMBURI": "ELIAS NTHIGA NDWIGA",
    "NYALI BAZAAR": "ELIAS NTHIGA NDWIGA",
    "KILIFI": "ELIAS NTHIGA NDWIGA",
    "MALINDI": "ELIAS NTHIGA NDWIGA",
    "MALINDI HIGHWAY": "ELIAS NTHIGA NDWIGA",
    "MTWAPA": "ELIAS NTHIGA NDWIGA",
    "NYALI": "MICAH AYEKO",
    "BOMBOLULU": "ELIAS NTHIGA NDWIGA",
    "ELDORET REFERRAL": "ELISHA JUMA WAFULA",
    "ELGON VIEW ELDORET": "ELISHA JUMA WAFULA",
    "KAPSABET": "ELISHA JUMA WAFULA",
    "ZION MALL ELDORET": "ELISHA JUMA WAFULA",
    "BARAKA EXPRESS": "VICTOR MWAKA",
    "MOI AVENUE": "VICTOR MWAKA",
    "MWANZI ROAD": "FRANCIS NDEGWA KIRING'U",
    "OJIJO ROAD": "VICTOR MWAKA",
    "RUAI": "EMILY MORAA",
    "SAIKA": "EMILY MORAA",
    "KUBWA": "FRANCIS MURAGE GICHOBI",
    "NAROK": "FRANCIS MURAGE GICHOBI",
    "NDOGO": "FRANCIS MURAGE GICHOBI",
    "SAFARI CENTER NAIVASHA": "FRANCIS MURAGE GICHOBI",
    "WESTLANDS": "FRANCIS NDEGWA KIRING'U",
    "BUNGOMA": "FREDRICK OMONDI YADA",
    "EMBU": "HILDAH WARUKIRA",
    "EMBU PEARL CENTER": "HILDAH WARUKIRA",
    "MAIYAN MALL RONGAI": "IRENE NTHENYA MUTHOKA",
    "KAMAKIS": "ISABELA WAMBUI",
    "GITHURAI 44": "ISABELA WAMBUI",
    "CAPITAL CENTER": "JAMES KARUGU WAITHERA",
    "HAZINA": "JAMES KARUGU WAITHERA",
    "SOUTH C": "JAMES KARUGU WAITHERA",
    "NAKURU DOWNTOWN": "JANE NJERI KARANJA",
    "NAKURU MIDTOWN": "JANE NJERI KARANJA",
    "NAKURU SC": "JANE NJERI KARANJA",
    "HOMEGROUND": "JOEL SIMIYU",
    "NGONG TOWN 2": "JOEL SIMIYU",
    "NGONG TOWN 3": "JOEL SIMIYU",
    "MERU": "JOHN ALLAN KULIKYA",
    "KAKAMEGA": "JOSEPH NDETEI MAKAU",
    "CIATA MALL KIAMBU ROAD": "JULIET NDUTA MBURU",
    "GACHIE": "JULIET NDUTA MBURU",
    "KIAMBU TOWN": "JULIET NDUTA MBURU",
    "THINDIGUA": "JULIET NDUTA MBURU",
    "GITHURAI 45": "LINET ATIENO",
    "GITHURAI": "LINET ATIENO",
    "KAHAWA SUKARI": "LINET ATIENO",
    "KASARANI": "LINET ATIENO",
    "LUCKY SUMMER": "LINET ATIENO",
    "SOKONI": "MARGARET IRUNGU",
    "EMBAKASI NYAYO": "MERCY MUTANU",
    "IMARA DAIMA": "MERCY MUTANU",
    "SUPERMARKET KOBIL": "MERCY MUTANU",
    "EXPRESS EMBAKASI": "MERCY MUTANU",
    "UTAWALA": "MERCY MUTANU",
    "KISII": "OLIPHA NYANGAMI",
    "KISII HYPER": "OLIPHA NYANGAMI",
    "JUJA CITY": "PAUL GUCHU",
    "SPUR MALL RUIRU": "PAUL GUCHU",
    "TATU CITY": "PAUL GUCHU",
    "KITENGELA MALL": "PERIS WAVINYA KITAKA",
    "KITUI": "PERIS WAVINYA KITAKA",
    "OLD": "PERIS WAVINYA KITAKA",
    "MACHAKOS MBAITU": "PERIS WAVINYA KITAKA",
    "MACHAKOS": "PERIS WAVINYA KITAKA",
    "LAVINGTON CURVE": "SHAMITA MOHAMED",
    "RIRUTA": "SHAMITA MOHAMED",
    "AGA KHAN WALK": "STEPHEN OTIENO",
    "DEVELOPMENT HOUSE": "STEPHEN OTIENO",
    "LIFESTYLE CBD": "STEPHEN OTIENO",
    "MUINDI MBINGU": "STEPHEN OTIENO",
    "RONALD NGALA": "STEPHEN OTIENO",
    "DIGO": "VERONICA ANYONA WOCHETE",
    "LIKONI": "VERONICA ANYONA WOCHETE",
    "MWEMBE TAYARI": "VERONICA ANYONA WOCHETE",
    "TUDOR MOMBASA": "VERONICA ANYONA WOCHETE",
    "UKUNDA": "VERONICA ANYONA WOCHETE",
    "BURUBURU": "VERONICA MWENDE MAWEU",
    "EASTGATE": "VERONICA MWENDE MAWEU",
    "GREENSPAN": "VERONICA MWENDE MAWEU",
    "KOMAROCK": "VERONICA MWENDE MAWEU",
    "T-SQUARE BURUBURU": "VERONICA MWENDE MAWEU",
    "UMOJA": "VERONICA MWENDE MAWEU",
    "THIKA TOWN": "MARGARET NJERI",
    "THIKA ANANAS": "MARGARET NJERI",
    "NYERI": "VERONICA WACHUKA"
}

# --- 2️⃣ Load Categorized file ---
file_path = "categorized_file.xlsx"  # Update path if needed
df = pd.read_excel(file_path)

# Convert to string, strip leading/trailing spaces
df['Branch'] = df['Branch'].astype(str).str.strip().str.upper()

# Replace multiple spaces with single space
df['Branch'] = df['Branch'].str.replace(r'\s+', ' ', regex=True)

# Optional: remove non-breaking spaces or other invisible characters
df['Branch'] = df['Branch'].str.replace(u'\xa0', ' ')

# --- 4️⃣ Map Handler to each Branch ---
df['Handler'] = df['Branch'].map(mapping_dict)

# Fill missing handlers with a placeholder
df['Handler'] = df['Handler'].fillna("UNKNOWN HANDLER")

# --- 5️⃣ Save cleaned file ---
output_file = "Branch_clean.xlsx"
df.to_excel(output_file, index=False)

print(f"✅ Branch_clean.xlsx created successfully!")


✅ Branch_clean.xlsx created successfully!


In [16]:
import pandas as pd

# --- Dictionary mapping branches to regions ---
branch_to_region = {
    # --- Nairobi ---
    "AGA KHAN WALK": "Nairobi",
    "AIRPORT VIEW": "Nairobi",
    "BARAKA EXPRESS": "Nairobi",
    "BURUBURU": "Nairobi",
    "CAPITAL CENTER": "Nairobi",
    "CIATA MALL KIAMBU ROAD": "Nairobi",
    "DEVELOPMENT HOUSE": "Nairobi",
    "EASTGATE": "Nairobi",
    "EMBAKASI NYAYO": "Nairobi",
    "EXPRESS EMBAKASI": "Nairobi",
    "GATEWAY MALL-SYOKIMAU": "Nairobi",
    "GITHURAI": "Nairobi",
    "GITHURAI 44": "Nairobi",
    "GITHURAI 45": "Nairobi",
    "GREENHOUSE": "Nairobi",
    "GREENSPAN": "Nairobi",
    "HAZINA": "Nairobi",
    "HOMEGROUND": "Nairobi",
    "HQ DC": "Nairobi",  # ✅ added new
    "IMARA DAIMA": "Nairobi",
    "JOGOO ROAD": "Nairobi",
    "JUJA CITY": "Nairobi",
    "KAHAWA SUKARI": "Nairobi",
    "KAMAKIS": "Nairobi",
    "KANGEMI": "Nairobi",
    "KASARANI": "Nairobi",
    "KATANI ROAD": "Nairobi",
    "KILIMANI": "Nairobi",
    "KINGARA ROAD": "Nairobi",
    "KOMAROCK": "Nairobi",
    "KUBWA": "Nairobi",
    "LANGATA": "Nairobi",
    "LANGATA MEDLINK": "Nairobi",
    "LAVINGTON CURVE": "Nairobi",
    "LIFESTYLE CBD": "Nairobi",
    "LUCKY SUMMER": "Nairobi",
    "MOI AVENUE": "Nairobi",
    "MOUNTAIN MALL": "Nairobi",
    "MOUNTAIN VIEW": "Nairobi",
    "MUINDI MBINGU": "Nairobi",
    "MWANZI ROAD": "Nairobi",
    "NAIROBI WEST": "Nairobi",
    "NDOGO": "Nairobi",
    "NGONG TOWN 2": "Nairobi",
    "NGONG TOWN 3": "Nairobi",
    "OJIJO ROAD": "Nairobi",
    "ONE STOP KAREN": "Nairobi",
    "PRESTIGE": "Nairobi",
    "RIRUTA": "Nairobi",
    "RONALD NGALA": "Nairobi",
    "RUAI": "Nairobi",
    "RUAKA": "Nairobi",
    "SAIKA": "Nairobi",
    "SHELL SURVEY THIKAROAD": "Nairobi",
    "SOKONI": "Nairobi",
    "SOUTH C": "Nairobi",
    "SPUR MALL RUIRU": "Nairobi",
    "TASSIA": "Nairobi",
    "TATU CITY": "Nairobi",
    "THINDIGUA": "Nairobi",
    "TILISI": "Nairobi",
    "T-SQUARE BURUBURU": "Nairobi",
    "UMOJA": "Nairobi",
    "UTAWALA": "Nairobi",
    "UTHIRU EXPRESS": "Nairobi",
    "WATERFRONT KAREN": "Nairobi",
    "WESTLANDS": "Nairobi",
    "WOOD AVENUE KILIMANI": "Nairobi",
    "MAGADI ROAD": "Nairobi",
    "MIHANGO": "Nairobi",
    "WESTBAY MALL REDHILL": "Nairobi",

    # --- Central ---
    "KIAMBU TOWN": "Central",
    "LIMURU": "Central",
    "THIKA ANANAS": "Central",
    "THIKA TOWN": "Central",
    "NYERI": "Central",

    # --- Eastern ---
    "EMBU": "Eastern",
    "EMBU PEARL CENTER": "Eastern",
    "KITUI": "Eastern",
    "MACHAKOS": "Eastern",
    "MACHAKOS MBAITU": "Eastern",
    "MAVOKO EXPRESS": "Eastern",
    "MERU": "Eastern",
    "ATHI RIVER": "Eastern",

    # --- Mombasa / Coastal ---
    "BAMBURI": "Mombasa/Coastal",
    "BOMBOLULU": "Mombasa/Coastal",
    "DIGO": "Mombasa/Coastal",
    "KILIFI": "Mombasa/Coastal",
    "LIKONI": "Mombasa/Coastal",
    "MALINDI": "Mombasa/Coastal",
    "MALINDI HIGHWAY": "Mombasa/Coastal",
    "MTWAPA": "Mombasa/Coastal",
    "MWEMBE TAYARI": "Mombasa/Coastal",
    "NYALI": "Mombasa/Coastal",
    "NYALI BAZAAR": "Mombasa/Coastal",
    "TUDOR MOMBASA": "Mombasa/Coastal",
    "UKUNDA": "Mombasa/Coastal",

    # --- Lower Rift ---
    "NAKURU DOWNTOWN": "Lower Rift",
    "NAKURU MIDTOWN": "Lower Rift",
    "NAKURU SC": "Lower Rift",
    "NAKURU WESTSIDE": "Lower Rift",
    "SAFARI CENTER NAIVASHA": "Lower Rift",
    "NAROK": "Lower Rift",
    "KITENGELA MALL": "Lower Rift",
    "MAIYAN MALL RONGAI": "Lower Rift",

    # --- Upper Rift ---
    "ELDORET REFERRAL": "Upper Rift",
    "ELGON VIEW ELDORET": "Upper Rift",
    "ZION MALL ELDORET": "Upper Rift",
    "KAPSABET": "Upper Rift",
    "KERICHO": "Upper Rift",

    # --- Nyanza ---
    "KISII": "Nyanza",
    "KISII HYPER": "Nyanza",
    "KISUMU": "Nyanza",
    "KISUMU MEGA CITY": "Nyanza",
    "KISUMU SIMBA": "Nyanza",

    # --- Western ---
    "BUNGOMA": "Western",
    "KAKAMEGA": "Western",
}

# --- Load Excel file ---
df = pd.read_excel("Branch_clean.xlsx")

# --- Clean column names and branch entries ---
# Convert to string, strip leading/trailing spaces
df['Branch'] = df['Branch'].astype(str).str.strip().str.upper()

# Replace multiple spaces with single space
df['Branch'] = df['Branch'].str.replace(r'\s+', ' ', regex=True)

# Optional: remove non-breaking spaces or other invisible characters
df['Branch'] = df['Branch'].str.replace(u'\xa0', ' ')

# --- Add regions ---
df["Region"] = df["Branch"].map(branch_to_region).fillna("Unclassified")

# --- Show unclassified branches ---
unclassified = df[df["Region"] == "Unclassified"]["Branch"].unique()
if len(unclassified) > 0:
    print("⚠️ Unclassified branches found:")
    for b in unclassified:
        print("-", b)
else:
    print("✅ All branches classified correctly.")

# --- Save output ---
df.to_excel("final_data.xlsx", index=False)
print("✅ Regions added and saved successfully.")

⚠️ Unclassified branches found:
- RIRONI
✅ Regions added and saved successfully.
